In [1]:
import json
import os
from dotenv import load_dotenv
from groq import Groq

In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found in .env"
    )

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq client initialized.")

Groq client initialized.


In [3]:
# load evidence packet
EVIDENCE_PATH = "../output/evidence_packet.json"

with open(
    EVIDENCE_PATH,
    "r",
    encoding="utf-8"
) as f:
    evidence_packet = json.load(f)

print("Evidence packet loaded.")

Evidence packet loaded.


In [4]:
evidence_json = json.dumps(
    evidence_packet,
    indent=2,
    ensure_ascii=False
)

print(
    evidence_json[:3000]
)

{
  "document_type": "Periodic Adverse Drug Experience Report",
  "product": {
    "name": "Bisoprolol"
  },
  "reporting_period": {
    "start": "2024-12-27",
    "end": "2025-12-26"
  },
  "dataset": {
    "reaction_records": 1068,
    "unique_cases": 1024
  },
  "case_summary": {
    "total_cases": 1024,
    "serious_cases": 1023,
    "non_serious_cases": 1,
    "expedited_cases": 1023,
    "fatal_cases": 68
  },
  "demographics": {
    "age_distribution": {
      "76+": 379,
      "61-75": 362,
      "46-60": 134,
      "Unknown": 87,
      "31-45": 36,
      "<18": 16,
      "18-30": 10
    },
    "age_percentage": {
      "76+": 37.01,
      "61-75": 35.35,
      "46-60": 13.09,
      "Unknown": 8.5,
      "31-45": 3.52,
      "<18": 1.56,
      "18-30": 0.98
    },
    "sex_distribution": {
      "female": 503,
      "male": 493,
      "NaN": 28
    },
    "sex_percentage": {
      "female": 49.12,
      "male": 48.14,
      "NaN": 2.73
    },
    "country_distribution": {
     

In [5]:
# system prompt
SYSTEM_PROMPT = """
You are a pharmacovigilance reporting assistant.

Your task is to draft a Periodic Adverse Drug Experience
Report (PADER) narrative using ONLY the evidence provided
by the deterministic analysis.

STRICT RULES:

1. Do not invent statistics, cases, dates, reactions,
   outcomes, percentages, or clinical facts.

2. Do not calculate new statistics yourself.
   Use the supplied deterministic results.

3. Do not interpret unknown or missing values as known.

4. Do not interpret the numeric age-unit code 800.
   Treat those cases as Unknown age.

5. Preserve country values exactly as supplied.

6. Do not claim that a reported adverse event was caused
   by bisoprolol merely because it appears in the dataset.

7. Use terms such as "reported cases", "reported reactions",
   and "observed in the dataset" where appropriate.

8. If evidence is insufficient for a conclusion, explicitly
   state that the available dataset does not support that
   conclusion.

9. Outcome percentages may not sum to 100% because a case
   can contain multiple reactions/outcomes.

10. The deterministic analysis is the source of truth for
    numerical information.

11. Do not introduce information from your general medical
    knowledge.

12. Produce a professional pharmacovigilance-style report.

The report should contain:

- Executive Summary
- Reporting Period
- Case Overview
- Seriousness Analysis
- Expedited Case Analysis
- Demographic Analysis
- Adverse Reaction Analysis
- Serious Reaction Analysis
- Fatal Case Analysis
- Outcome Analysis
- Geographic Distribution
- Temporal Trends
- Limitations
- Conclusion
"""

In [6]:
# user prompt
USER_PROMPT = f"""
Prepare a PADER draft from the following deterministic
evidence packet.

EVIDENCE PACKET:

{evidence_json}

Write the report using only the supplied evidence.

Important:
The output is a draft for review. Do not fabricate or
infer information that is not explicitly supported by
the evidence packet.
"""

In [7]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT
        }
    ],
    temperature=0.1,
    max_tokens=8000
)

pader_draft = response.choices[0].message.content

print(pader_draft)

**Periodic Adverse Drug Experience Report (PADER) Draft**

**Executive Summary**

This report summarizes the adverse drug experience data for Bisoprolol, as reported during the period from December 27, 2024, to December 26, 2025. The dataset contains 1024 unique cases, with 1023 reported as serious and 68 reported as fatal. The most frequently reported reactions include Acute kidney injury, Drug ineffective, and Hypotension.

**Reporting Period**

The reporting period for this analysis is from December 27, 2024, to December 26, 2025.

**Case Overview**

A total of 1024 unique cases were reported during the analysis period, with 1023 (99.9%) of these cases reported as serious and 1 (0.1%) reported as non-serious. Additionally, 1023 (99.9%) cases were reported as expedited.

**Seriousness Analysis**

Of the 1024 total cases, 1023 (99.9%) were reported as serious. The seriousness criteria for these cases include death (67), life-threatening (105), hospitalization (480), disabling (43), co

In [8]:
# save pader draft
PADER_PATH = "../output/pader_draft.md"

with open(
    PADER_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(pader_draft)

print(
    "PADER draft saved to:",
    PADER_PATH
)

PADER draft saved to: ../output/pader_draft.md


In [9]:
# final report
print(
    "=" * 80
)

print(
    "PADER DRAFT"
)

print(
    "=" * 80
)

print(
    pader_draft
)

PADER DRAFT
**Periodic Adverse Drug Experience Report (PADER) Draft**

**Executive Summary**

This report summarizes the adverse drug experience data for Bisoprolol, as reported during the period from December 27, 2024, to December 26, 2025. The dataset contains 1024 unique cases, with 1023 reported as serious and 68 reported as fatal. The most frequently reported reactions include Acute kidney injury, Drug ineffective, and Hypotension.

**Reporting Period**

The reporting period for this analysis is from December 27, 2024, to December 26, 2025.

**Case Overview**

A total of 1024 unique cases were reported during the analysis period, with 1023 (99.9%) of these cases reported as serious and 1 (0.1%) reported as non-serious. Additionally, 1023 (99.9%) cases were reported as expedited.

**Seriousness Analysis**

Of the 1024 total cases, 1023 (99.9%) were reported as serious. The seriousness criteria for these cases include death (67), life-threatening (105), hospitalization (480), disabl